# Employee Wellness Analysis — Exploratory Data Analysis### XYZ Technical Solutions — Mental Health in the Workplace**Objective:** Identify employees who may need mental-health support and uncover the workplacefactors (benefits, company size, remote work, family history, etc.) that are associated withseeking treatment, so HR can design targeted wellness programs.

## 1. Understanding the Dataset

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (8,5)df = pd.read_csv('employee_wellness_dataset.csv')print('Shape:', df.shape)df.head()

In [ ]:
df.info()

In [ ]:
# Missing values per columndf.isna().sum().sort_values(ascending=False).head(10)

**Observations:**- 1048 responses, 28 columns (mostly categorical survey answers).- `comments` (916 missing) and `state` (412 missing) are heavily empty — expected, since  `state` only applies to US respondents and `comments` was optional free text.- `work_interfere` has 236 missing values — these are respondents without a diagnosed  mental health condition, so the question didn't apply to them.- `Age` and `Gender` contain **data-entry noise** that must be cleaned before analysis.

## 2. Data Preprocessing

In [ ]:
# --- Age: remove impossible values (negative ages, typos like 99999999999, 329) ---print(sorted(df['Age'].unique())[:5], sorted(df['Age'].unique())[-5:])df['Age_clean'] = df['Age'].apply(lambda x: x if 18 <= x <= 75 else np.nan)print('Outliers removed:', df['Age'].apply(lambda x: not (18 <= x <= 75)).sum())print(df['Age_clean'].describe())

In [ ]:
# --- Gender: 30+ free-text spellings collapsed into 3 clean categories ---print(df['Gender'].str.lower().unique())def clean_gender(g):    g = str(g).strip().lower()    male_set = {'male','m','male-ish','maile','cis male','mal','male (cis)','make',                'man','msle','mail','malr','cis man','guy (-ish) ^_^',                'male leaning androgynous'}    female_set = {'female','f','cis female','woman','femake','female (cis)',                  'female (trans)','cis-female/femme','trans woman','trans-female'}    if g in male_set:        return 'Male'    if g in female_set:        return 'Female'    return 'Other/Non-binary'df['Gender_clean'] = df['Gender'].apply(clean_gender)df['Gender_clean'].value_counts()

In [ ]:
# --- no_employees: Excel auto-corrupted two category labels into dates ---# '25-Jun' was originally '6-25' employees, '5-Jan' was originally '1-5' employeesprint(df['no_employees'].unique())map_fix = {'25-Jun':'6-25', '5-Jan':'1-5'}df['no_employees_clean'] = df['no_employees'].replace(map_fix)order_size = ['1-5','6-25','26-100','100-500','500-1000','More than 1000']df['no_employees_clean'] = pd.Categorical(df['no_employees_clean'], categories=order_size, ordered=True)

In [ ]:
# --- Missing value handling ---df['self_employed'] = df['self_employed'].replace('NA', np.nan).fillna('No')   # modedf['work_interfere_clean'] = df['work_interfere'].fillna('Not Applicable')     # no condition reporteddf = df.drop(columns=['comments'])   # too sparse (87% missing) to useprint('Duplicate rows:', df.duplicated().sum())df.shape

## 3. Exploratory Data Analysis

### 3.1 Age Distribution

In [ ]:
bins = [17,24,29,34,39,44,49,75]labels = ['18-24','25-29','30-34','35-39','40-44','45-49','50+']df['age_bin'] = pd.cut(df['Age_clean'], bins=bins, labels=labels)ax = df['age_bin'].value_counts().reindex(labels).plot(kind='bar', color='#028090')ax.set_title('Age Distribution of Respondents')ax.set_xlabel('Age Group'); ax.set_ylabel('Count')plt.tight_layout(); plt.show()

**Insight:** The workforce sampled skews young — roughly 60% of respondents are between 25 and 34, a career stage often associated with high workload and career-establishment stress.

### 3.2 Gender Distribution

In [ ]:
ax = df['Gender_clean'].value_counts().plot(kind='bar', color='#00A896')ax.set_title('Gender Distribution'); ax.set_xlabel(''); ax.set_ylabel('Count')plt.tight_layout(); plt.show()

**Insight:** The sample is heavily male (~79%), so gender-based comparisons should be interpreted cautiously due to sample-size imbalance.

### 3.3 Overall Treatment-Seeking Rate

In [ ]:
ax = df['treatment'].value_counts().plot(kind='pie', autopct='%1.1f%%',        colors=['#B2DBBF','#028090'], ylabel='')ax.set_title('Has the Employee Sought Mental Health Treatment?')plt.tight_layout(); plt.show()

**Insight:** Nearly half (48.9%) of respondents have already sought treatment for a mental health condition — this is the group whose experience should shape wellness-program design.

### 3.4 Family History vs Treatment

In [ ]:
ct = pd.crosstab(df['family_history'], df['treatment'], normalize='index')*100ax = ct.plot(kind='bar', stacked=False, color=['#F2A65A','#028090'])ax.set_title('Treatment Rate by Family History of Mental Illness')ax.set_ylabel('% of group'); ax.set_xlabel('Family History')plt.tight_layout(); plt.show()

**Insight:** Employees with a family history of mental illness seek treatment at **57.8%**, nearly 3x the rate of those without one (**20.3%**) — the single strongest predictor in the dataset.

### 3.5 Work Interference vs Treatment

In [ ]:
order_wi = ['Never','Rarely','Sometimes','Often','Not Applicable']ct = pd.crosstab(df['work_interfere_clean'], df['treatment']).reindex(order_wi)ax = ct.plot(kind='bar', color=['#F2A65A','#028090'])ax.set_title('Treatment by Reported Work Interference')ax.set_ylabel('Count'); ax.set_xlabel('')plt.tight_layout(); plt.show()

**Insight:** Respondents who say their condition interferes with work 'Sometimes' or 'Often' overwhelmingly seek treatment, confirming self-reported interference is a strong, actionable warning sign.

### 3.6 Company Size vs Treatment

In [ ]:
ct = pd.crosstab(df['no_employees_clean'], df['treatment'])ax = ct.plot(kind='bar', color=['#F2A65A','#028090'])ax.set_title('Treatment by Company Size'); ax.set_ylabel('Count'); ax.set_xlabel('')plt.tight_layout(); plt.show()

**Insight:** Treatment-seeking is fairly stable across company sizes, with a slightly higher share of small companies (1-5 employees) reporting treatment — possibly reflecting less formal support infrastructure.

### 3.7 Employer Benefits vs Treatment

In [ ]:
ct = pd.crosstab(df['benefits'], df['treatment'])ax = ct.plot(kind='bar', color=['#F2A65A','#028090'])ax.set_title('Treatment by Awareness of Mental Health Benefits')ax.set_ylabel('Count'); ax.set_xlabel('')plt.tight_layout(); plt.show()

**Insight:** Employees who know their employer offers mental-health benefits seek treatment at **62%**, versus only **36%** among those who say 'Don't know' — clear communication of existing benefits matters as much as the benefits themselves.

### 3.8 Top Countries Represented

In [ ]:
top_countries = df['Country'].value_counts().head(8)ax = top_countries.plot(kind='barh', color='#028090')ax.set_title('Top 8 Countries by Respondent Count')ax.invert_yaxis()plt.tight_layout(); plt.show()

**Insight:** The US (61%) and UK (15%) dominate the sample, so region-specific policy differences (e.g. healthcare access) may be confounding some of the patterns above.

## 4. Key Insights & Interpretation1. **Family history is the strongest signal** — employees with a family history of mental illness seek treatment at nearly 3x the rate of those without.2. **Self-reported work interference tracks closely with treatment** — this is a practical early-warning question managers/HR can ask.3. **Awareness of benefits drives usage** — treatment-seeking is highest among employees who *know* mental-health benefits exist, not necessarily where benefits exist but are poorly communicated.4. **Roughly half of employees have already sought treatment**, confirming this is a mainstream, not fringe, workplace issue.5. **Company size and remote-work status show only mild effects**, suggesting culture and communication matter more than structural factors.6. The sample skews **young and male**, and **US/UK-heavy** — conclusions should be validated against XYZ's own workforce demographics before rollout.## 5. Recommendations for XYZ Technical Solutions- Proactively **communicate existing mental-health benefits and care options** — awareness alone raised treatment-seeking by up to 26 percentage points.- Train managers to recognise **self-reported work-interference language** as a cue to check in and offer support.- Given nearly half of employees report needing treatment, build **manager-facing wellness check-ins** rather than opt-in-only programs.- Consider **anonymous, confidential** channels since willingness to disclose to supervisors/coworkers varies widely.- Re-run this analysis on XYZ's own employee survey data (not just this public reference dataset) before finalizing programs.